# ASG-WM / FaithCast — **Train All**

Runs the full training pipeline end-to-end and produces **`train_results.zip`** (checkpoints + cached ASG labels + result tables).

**Pipeline:** pre-flight → download → auto-label → Tier-0 → Tier-1 (VLM curriculum) → Tier-2 → zip.

Designed for Colab A100 (`Runtime → Run all`). Runs on a laptop/CPU too with `DATASET="synthetic"` for a wiring smoke test. See `RUN_PLAN.md` for the decision gates.

In [ ]:
# --- 1. Locate / clone the repo ----------------------------------------------
# On Colab: set ASGWM_REPO_URL to your git remote, this cell clones it.
# Locally: just run this notebook from the repo root (the cell is a no-op).
import os, sys, subprocess, pathlib

REPO_URL = os.environ.get("ASGWM_REPO_URL", "")          # e.g. https://github.com/<you>/<repo>.git
REPO_DIR = "Forecasting-Through-Meteorological-Reasoning"

if not pathlib.Path("src/asgwm").exists():
    if not pathlib.Path(REPO_DIR, "src", "asgwm").exists():
        assert REPO_URL, "Set ASGWM_REPO_URL (Colab) or launch this notebook from the repo root."
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

assert pathlib.Path("src/asgwm").exists(), f"Run from repo root; cwd={os.getcwd()}"
print("repo root:", os.getcwd())


In [ ]:
# --- 2. Install dependencies --------------------------------------------------
# Core package + pinned requirements. Real-data extras are per-dataset (uncomment).
get_ipython().system('pip -q install -e ./src')
get_ipython().system('pip -q install -r src/requirements.txt')

# Real-data backends (only needed when DATASET != "synthetic"):
#   SEVIR  : pip -q install s3fs h5py
#   MRMS   : pip -q install boto3 cfgrib eccodes xarray
#   NEXRAD : pip -q install arm-pyart boto3


In [ ]:
# --- 3. Run configuration -----------------------------------------------------
import subprocess, sys, itertools

DATASET      = "synthetic"   # "sevir" | "synthetic" | "nexrad" | "mrms"
REQUIRE_REAL = False         # True => a real-data load failure RAISES (use for paid runs)
N_EVENTS     = 64            # raise to 800+ for real runs (RUN_PLAN.md)
FIRST_RUN    = True          # time-boxed step caps that fit ~2 A100 sessions
CFG          = "src/configs/default.yaml"

def ov(**kw):
    return list(itertools.chain.from_iterable(["--override", f"{k}={v}"] for k, v in kw.items()))

COMMON = ov(**{
    "data.dataset": DATASET,
    "data.require_real": str(REQUIRE_REAL).lower(),
    "data.n_train_events": N_EVENTS,
})

def run(script, extra=None):
    cmd = [sys.executable, f"src/scripts/{script}", "--config", CFG] + COMMON + (extra or [])
    print(">>", " ".join(cmd)); sys.stdout.flush()
    subprocess.run(cmd, check=True)

print(f"dataset={DATASET}  require_real={REQUIRE_REAL}  n_events={N_EVENTS}  first_run={FIRST_RUN}")


In [ ]:
# --- 4. Pre-flight GO/NO-GO (RUN_PLAN.md gates 1-2) ---------------------------
# Catches data/loader/GPU/OOM problems in ~2 min instead of mid-run. Must print GO.
run("99_preflight.py")


In [ ]:
# --- 5. Download / cache the dataset -----------------------------------------
# Idempotent: re-running skips events already cached under artifacts/cache/events.
run("00_download_data.py")


In [ ]:
# --- 6. Auto-label ASGs (CPU; cached once, reused every session) -------------
run("01_autolabel.py")


In [ ]:
# --- 7. Tier-0: transition transformer + deterministic renderer --------------
extra = ov(**{"train.tier0.max_steps": 4000}) if FIRST_RUN else []
run("10_train_tier0.py", extra)


In [ ]:
# --- 8. Tier-1: VLM five-phase curriculum (hard Ph-3 ASG-F1 gate) ------------
extra = ov(**{"train.tier1.max_steps_per_phase": 4000}) if FIRST_RUN else []
run("20_train_tier1_curriculum.py", extra)


In [ ]:
# --- 9. Tier-2: end-to-end with intervention consistency ---------------------
extra = ov(**{"train.tier2.max_steps": 12000}) if FIRST_RUN else []
run("30_train_tier2.py", extra)


In [ ]:
# --- 10. Bundle everything needed to resume / evaluate -> train_results.zip ---
import glob, os, zipfile

OUT = "train_results.zip"
ROOTS = ["artifacts/ckpt", "artifacts/results", "artifacts/cache/asg"]
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
    for root in ROOTS:
        for p in glob.glob(os.path.join(root, "**", "*"), recursive=True):
            if os.path.isfile(p):
                z.write(p)
size_mb = os.path.getsize(OUT) / 1e6 if os.path.exists(OUT) else 0.0
print(f"wrote {OUT}  ({size_mb:.1f} MB)  -> download this to keep checkpoints + labels + tables")


**Done.** Download `train_results.zip`. Next: run `eval.ipynb` to produce the paper tables and figures.